### Importing Libaries

In [ ]:
import numpy as np
import pandas as pd
import os

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from imblearn.over_sampling import SMOTE

from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
from sklearn.compose import ColumnTransformer

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

#### Loading Data

In [ ]:
path = os.path.join('..', 'Data', 'creditcard.csv')
data = pd.read_csv(path)

In [ ]:
data.head()

In [ ]:
data.isna().sum()

In [ ]:
train, test = train_test_split(data, test_size = 0.2, random_state = 42)

In [ ]:
y_true = test['Class']
test.drop(['Class'], axis = 1, inplace = True)

#### Preprocessing

In [ ]:
X = train.drop(columns = ['Class'])
y = train['Class']

In [ ]:
X.head()

In [ ]:
cols = ['Time', 'Amount']

In [ ]:
preprocessor = ColumnTransformer(transformers = [
    ('Scaler', StandardScaler(), cols)
],
    remainder = 'passthrough')

In [ ]:
X_scaled = preprocessor.fit_transform(X)
test = preprocessor.transform(test)

In [ ]:
sm = SMOTE(random_state = 42)
X_resampled, y_resampled = sm.fit_resample(X_scaled, y)

In [ ]:
models = {
    'LogisticRegression': LogisticRegression(class_weight = 'balanced'),
    'RandomForestClassifier': RandomForestClassifier(class_weight = 'balanced'),
    'DecisionTreeClassifier': DecisionTreeClassifier(),
    'XGBClassifier': XGBClassifier(),
    'CatBooostClassifier': CatBoostClassifier(verbose = False)
}

In [ ]:
for i in range(len(models)):
    model = list(models.values())[i]

    model.fit(X_resampled, y_resampled)
    y_pred = model.predict(test)
    y_prob = model.predict_proba(test)[:, 1]

    print('*' * 20)
    print(f'Model: {list(models.keys())[i]}')
    print(f'Precision: {precision_score(y_true, y_pred)}')    
    print(f'Recall Score: {recall_score(y_true, y_pred)}')    
    print(f'F1 Score: {f1_score(y_true, y_pred)}')    
    print(f'Confusion Matrix: \n{confusion_matrix(y_true, y_pred)}')
    print(f'ROC-AUC: {roc_auc_score(y_true, y_prob)}')

#### Feature importance

In [ ]:
cb = CatBoostClassifier(verbose = False)
cb.fit(X_scaled, y)

In [ ]:
feature_names = X.columns
imp_features = pd.Series(cb.get_feature_importance(), index = feature_names)

imp_features.sort_values(ascending = False).plot(kind = 'bar')